# 02 Exploration by agreement and topic

Use this notebook for analyst-driven exploration within the selected data agreement. Keep approvals in `01` and enforcement in `03_pc`.


## 01 Introduction and scope

Set the active agreement/topic/table for this exploration run.


In [ ]:
agreement_id = "REPLACE_AGREEMENT_ID"
topic = "REPLACE_TOPIC"
table_name = "REPLACE_TABLE_NAME_OR_ALIAS"


## 02 Agreement context

This notebook selects an agreement and captures read-only metadata context before dataset exploration.


## 03 Configuration and imports


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit import (
    read_lakehouse_table,
    read_lakehouse_csv,
    read_lakehouse_parquet,
    read_warehouse_table,
    profile_dataframe,
    load_dq_rules,
    draft_dq_rules,
    run_dq_rule_review_widget,
    get_dq_review_results,
    write_dq_rules,
    review_dq_rule_deactivations,
    select_agreement,
    get_selected_agreement,
    register_current_notebook,
    load_governance,
    load_notebook_registry,
)

metadata_store = CONFIG.path_config.paths[ENV_NAME]["metadata"]
dq_metadata_table = CONFIG.review_workflow_config.dq_approved_table


## 04 Metadata exploration


### A. Agreement selection


In [ ]:
select_agreement(CONFIG, ENV_NAME, spark_session=spark)


### B. Selected agreement context and notebook registration


In [ ]:
import json

selected_agreement = get_selected_agreement()
agreement_id = selected_agreement["agreement_id"]
contract_version = selected_agreement["contract_version"]
agreement_name = selected_agreement.get("agreement_name", agreement_id)
domain = selected_agreement.get("domain", "")
business_purpose = selected_agreement.get("business_purpose", "")
approved_usage = selected_agreement.get("approved_usage", "")
steward_id = selected_agreement.get("steward_id", "")
custom_fields = json.loads(selected_agreement.get("custom_fields_json") or "{}")

agreement_context = {
    "agreement_id": agreement_id,
    "contract_version": contract_version,
    "agreement_name": agreement_name,
    "domain": domain,
    "business_purpose": business_purpose,
    "approved_usage": approved_usage,
    "steward_id": steward_id,
    "custom_fields": custom_fields,
}
dataset_name = agreement_name

register_current_notebook(
    spark,
    metadata_path=metadata_store,
    agreement_id=agreement_id,
    notebook_type="02_ex",
    environment_name=ENV_NAME,
    dataset_name=dataset_name,
    table_name=table_name,
    topic=topic,
)


### C. Existing approved DQ rules (read-only)


In [ ]:
# existing_dq_df = read_lakehouse_table(CONFIG, ENV_NAME, "metadata", dq_metadata_table)
# approved_active_rules = load_dq_rules(existing_dq_df, table_name=table_name)
# print(f"existing active DQ rules: {len(approved_active_rules)}")


### D. Existing governance/classification metadata (read-only)


In [ ]:
# Replace these table names if your 00_env_config uses different metadata table names.
# These reads are optional and read-only.
governance_context = {"columns": []}
try:
    governance_df = read_lakehouse_table(CONFIG, ENV_NAME, "metadata", "METADATA_COLUMN_GOVERNANCE")
    agreement_df = read_lakehouse_table(CONFIG, ENV_NAME, "metadata", "METADATA_DATA_AGREEMENT")
    governance_context = load_governance(
        governance_df,
        agreement_rows=agreement_df,
        agreement_id=agreement_id,
        table_name=table_name,
    )
except Exception as exc:
    governance_context = {"columns": []}
    print(f"No governance context loaded: {exc}")


### E. Existing notebook registry / prior evidence (read-only)


In [ ]:
try:
    existing_dq_df = read_lakehouse_table(CONFIG, ENV_NAME, "metadata", dq_metadata_table)
    approved_active_rules = load_dq_rules(existing_dq_df, table_name=table_name)
except Exception as exc:
    approved_active_rules = []
    print(f"No existing DQ rules loaded: {exc}")

try:
    notebook_registry_rows = load_notebook_registry(spark, agreement_id=agreement_id)
except Exception as exc:
    notebook_registry_rows = []
    print(f"No notebook registry rows loaded: {exc}")

print(f"number of governance columns: {len(governance_context.get('columns', []))}")
print(f"number of approved DQ rules: {len(approved_active_rules)}")
print(f"number of registry rows: {len(notebook_registry_rows)}")

display(agreement_context)
display(governance_context)
display(approved_active_rules)
display(notebook_registry_rows)


## 05 Dataset exploration


In [ ]:
# Option A: Lakehouse Delta table
source_df = read_lakehouse_table(CONFIG, ENV, "source", table_name, spark_session=spark)

# Option B: Warehouse table
# source_df = read_warehouse_table(CONFIG, ENV, "warehouse", "dbo", table_name, spark_session=spark)

# Option C: Lakehouse CSV file
# source_df = read_lakehouse_csv(CONFIG, ENV, "source", "Files/REPLACE_PATH.csv", spark_session=spark)

# Option D: Lakehouse Parquet file
# source_df = read_lakehouse_parquet(CONFIG, ENV, "source", "Files/REPLACE_PATH.parquet", spark_session=spark)

source_df.printSchema()
source_df.limit(20).show(truncate=False)


In [ ]:
profile_df = profile_dataframe(source_df, table_name=table_name)
profile_df.show(50, truncate=False)


In [ ]:
# Add focused exploratory checks here.
# Examples: null checks, distinct counts, joins, mappings, date logic, duplicate checks.
# Keep this exploratory. Final repeatable transformation logic belongs in 03_pc.


## 06 Findings

### Key findings

### Source quirks

### Business context notes

### Classification / sensitivity notes

### Suggested pipeline transform notes

### Open questions

### Handoff notes

- Approved DQ rules from section 07 are consumed by `03_pc`.
- Governance/classification updates follow the data agreement workflow.
- Production transformations and deterministic enforcement belong in `03_pc`.


In [ ]:
# Optional: add focused evidence snippets here.
# Record findings in markdown above.
# Do not define final DQ rules here.


## 07 AI-assisted DQ flow


In [ ]:
# 1) Load existing approved/active DQ rules
# existing_dq_df = read_lakehouse_table(CONFIG, ENV_NAME, "metadata", dq_metadata_table)
# approved_active_rules = load_dq_rules(existing_dq_df, table_name=table_name)

# 2) Generate AI candidate rules when needed
# candidate_rules = draft_dq_rules(
#     profile_df=profile_df,
#     table_name=table_name,
#     business_context=f"Business purpose: {business_purpose}\nApproved usage: {approved_usage}",
#     prompt_template=CONFIG.ai_prompt_config.dq_rule_suggestion_prompt_template,
# )

# 3) Review/edit/approve/reject using the DQ widget
# run_dq_rule_review_widget(candidate_rules, table_name=table_name)

# 4) Collect widget results
# review_results = get_dq_review_results(
#     table_name=table_name,
#     environment_name=ENV_NAME,
#     dataset_name=dataset_name,
# )

# approved_rules = review_results.get("approved_rules", [])

# 5) Persist only approved rules
# if approved_rules:
#     write_dq_rules(
#         approved_rules,
#         table_name=table_name,
#         metadata_path=metadata_store,
#     )

# 6) Review existing active rules for deactivation
# deactivation_reviews = review_dq_rule_deactivations(
#     approved_active_rules,
#     table_name=table_name,
# )

# 7) Persist deactivation metadata through an approved public writer when enabled.
# Keep exploratory notebooks on public package APIs; do not import internal builders.
